# Native PyTorch tutorial 3

600 epochs on all 191,775 cylinder-flow samples, area-weighted loss and actual gradient centralization. The input ordering is [t,x,y].

Run all cells in order. Full scientific workloads are the default; `NIF_SMOKE=1` is a separate development run and does not establish full convergence. The original TensorFlow and `_colab` notebooks are preserved as legacy references.

In [ ]:
# Run this cell first in a clean Colab Python 3.11–3.13 session.
import os, sys, subprocess, importlib.util
from pathlib import Path
# A local checkout is used directly for development; Colab installs the public pytorch branch.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "nif" / "model.py").exists()), None)
if repo is not None:
    sys.path.insert(0, str(repo))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        "torch==2.14.1", "numpy==2.1.3", "scipy==1.16.3", "matplotlib==3.10.0",
        "protobuf==7.36.2", "google-crc32c==1.9.0", "tfrecord==1.14.6",
        "git+https://github.com/pswpswpsw/nif.git@pytorch"])
import copy, json, time, math
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from matplotlib import pyplot as plt
import nif
# FULL is the default. Set NIF_SMOKE=1 before starting only for a short development check.
SMOKE = os.environ.get("NIF_SMOKE", "0") == "1"
device = torch.device(os.environ.get("NIF_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
torch.manual_seed(42)
np.random.seed(42)
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print({"torch": torch.__version__, "device": str(device), "mode": "SMOKE" if SMOKE else "FULL"})

OUT = Path(os.environ.get("NIF_OUTPUT_DIR", "pytorch_outputs")) / "tutorial3"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Explicit native loops replace compile/fit/callbacks. The loss reduction is per row,
# then the mean over the batch; area weights multiply each row's MSE (not weight-normalized).
def dense_tanh(inputs, outputs):
    # Preserve the original Dense default: Glorot uniform kernel and zero bias.
    layer = nn.Linear(inputs, outputs)
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)
    return nn.Sequential(layer, nn.Tanh())

def epochs(full):
    return 2 if SMOKE else full

def tensor(array, on_device=device):
    return torch.as_tensor(array, dtype=torch.float32, device=on_device)

def loader(features, target, batch_size, weight=None):
    arrays = [tensor(features, "cpu"), tensor(target, "cpu")]
    if weight is not None:
        arrays.append(tensor(weight, "cpu").reshape(-1))
    return DataLoader(TensorDataset(*arrays), batch_size=batch_size, shuffle=True,
                      drop_last=False, pin_memory=device.type == "cuda")

def predict(model, features, batch_size=1024):
    model.eval()
    result = []
    with torch.no_grad():
        for xb in torch.as_tensor(features, dtype=torch.float32).split(batch_size):
            result.append(model(xb.to(next(model.parameters()).device)).detach().cpu())
    return torch.cat(result).numpy()

def centralize_gradients(model):
    # TF kernels are [input,output]; native Linear stores [output,input].
    # Centralize along the input axes, preserving the original mathematical operation.
    for module in model.modules():
        for name, parameter in module.named_parameters(recurse=False):
            if parameter.grad is not None and parameter.ndim > 1:
                axes = tuple(range(1, parameter.ndim)) if isinstance(module, nn.Linear) else tuple(range(parameter.ndim-1))
                parameter.grad.sub_(parameter.grad.mean(dim=axes, keepdim=True))

def save_checkpoint(model, optimizer, path, epoch):
    payload = {"model_state": {k: v.detach().cpu() for k,v in model.state_dict().items()},
               "optimizer_state": optimizer.state_dict(), "epoch": epoch,
               "framework": "pytorch", "smoke": SMOKE}
    if hasattr(model, "cfg_shape_net"):
        payload.update(model_class=type(model).__name__, cfg_shape_net=model.cfg_shape_net,
                       cfg_parameter_net=model.cfg_parameter_net, mixed_policy="float32")
    torch.save(payload, path)

def save_and_check(model, optimizer, name, example, factory=None):
    path = OUT / (name + ".pt")
    save_checkpoint(model, optimizer, path, -1)
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if factory is None:
        factory = lambda: type(model)(copy.deepcopy(model.cfg_shape_net),
                                       copy.deepcopy(model.cfg_parameter_net), "float32")
    restored = factory().to(device)
    restored.load_state_dict(checkpoint["model_state"])
    restored.eval(); model.eval()
    with torch.no_grad():
        expected, actual = model(tensor(example)), restored(tensor(example))
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    print("Checkpoint predictions verified:", path)
    return restored

def train(model, batches, optimizer, nepoch, name, schedule=None,
          centralization=False, loss_function=None, before_epoch=None):
    history = []
    start = time.monotonic()
    for epoch in range(nepoch):
        model.train()
        if schedule is not None:
            for group in optimizer.param_groups:
                group["lr"] = schedule(epoch, group["lr"])
        if before_epoch is not None:
            before_epoch(epoch)
        summed, seen = {}, 0
        for batch in batches:
            xb, yb = (item.to(device) for item in batch[:2])
            optimizer.zero_grad(set_to_none=True)
            if loss_function is None:
                per_row = (model(xb) - yb).square().mean(dim=-1)
                if len(batch) == 3:
                    per_row = per_row * batch[2].to(device).reshape(-1)
                data_loss = per_row.mean()
                values = {"mse": data_loss}
            else:
                data_loss, values = loss_function(model, xb, yb)
            regularization = model.regularization_loss(xb) if hasattr(model, "regularization_loss") else xb.new_zeros(())
            loss = data_loss + regularization
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss in {name}, epoch {epoch}")
            loss.backward()
            if centralization:
                centralize_gradients(model)
            optimizer.step()
            for key, value in {**values, "regularization": regularization, "loss": loss}.items():
                summed[key] = summed.get(key, 0.0) + float(value.detach()) * len(xb)
            seen += len(xb)
        row = {"epoch": epoch, "rows": seen, **{key: value / seen for key,value in summed.items()}}
        history.append(row)
        if epoch % 100 == 0 or epoch == nepoch - 1:
            print(name, row, "elapsed_seconds", round(time.monotonic()-start, 2))
            (OUT / (name + "_history.json")).write_text(json.dumps(history, indent=2))
        if epoch % 1000 == 0 or epoch == nepoch - 1:
            save_checkpoint(model, optimizer, OUT / (name + "_latest.pt"), epoch)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.semilogy([row["loss"] for row in history]); ax.set(xlabel="epoch", ylabel="loss", title=name)
    fig.tight_layout(); fig.savefig(OUT / (name + "_loss.png")); plt.show(); plt.close(fig)
    return history

def wave_figure(truth, prediction, name):
    fig, axs = plt.subplots(1,3,figsize=(15,3))
    for ax, values, title in zip(axs, (truth,prediction,prediction-truth), ("true","pred","error")):
        im=ax.contourf(tt,xx,values.reshape(10,200),levels=50,cmap="seismic")
        fig.colorbar(im,ax=ax); ax.set(xlabel="t",ylabel="x",title=title)
    fig.tight_layout(); fig.savefig(OUT / (name+"_reconstruction.png")); plt.show(); plt.close(fig)

def write_metrics(values):
    print(json.dumps(values, indent=2))
    (OUT / "metrics.json").write_text(json.dumps({"mode": "smoke" if SMOKE else "full", **values}, indent=2))


# A simple flow over a cylinder dataset from Castro, an AMR solver

In [ ]:
from nif.demo import CylinderFlow
tmp = CylinderFlow()
train_data, sample_weight = tmp.data, tmp.sample_weight
assert len(train_data) == 191775
print(train_data.shape, sample_weight.shape, tmp.std)


# Try NIFMultiScale linear

In [ ]:
cfg_shape_net = {
    "use_resblock":False,
    "connectivity": 'last_layer',
    "input_dim": 2,
    "output_dim": 2,
    "units": 30,
    "nlayers": 2,
    "weight_init_factor": 0.01,
    "omega_0":30.0
}
cfg_parameter_net = {
    "use_resblock":False,
    "input_dim": 1,
    "latent_dim": 10,
    "units": 30,
    "nlayers": 2,
    "activation": 'swish'
}

In [ ]:
nepoch, lr, batch_size = 600, 1e-3, 1024
# Only smoke mode selects fewer training rows; FULL uses all 191,775 AMR points each epoch.
training_rows = 4096 if SMOKE else len(train_data)
train_dataset = loader(train_data[:training_rows,:3], train_data[:training_rows,3:5],
                       batch_size, sample_weight[:training_rows])
def scheduler(epoch, lr):
    if epoch < 200: return lr
    if epoch < 400: return 5e-4
    if epoch < 600: return 2e-4
    return 3e-5


# Try `NIFMultiScale`

In [ ]:
from nif.optimizers import AdaBeliefOptimizer
model_ori = nif.NIFMultiScaleLastLayerParameterized(cfg_shape_net,cfg_parameter_net,"float32").to(device)
model = model_ori.build()
optimizer = AdaBeliefOptimizer(model.parameters(),lr=lr)
history = train(model,train_dataset,optimizer,epochs(nepoch),"cylinder",schedule=scheduler,centralization=True)
model = save_and_check(model,optimizer,"cylinder",train_data[:64,:3])
model_ori = model


# draw latent representation

In [ ]:
model_p_to_lr = model_ori.model_p_to_lr()


In [ ]:
t_arr = np.linspace(-1,1,100)
with torch.no_grad():
    lr_arr = model_p_to_lr(tensor(t_arr[:,None])).cpu().numpy()


In [ ]:
fig,ax=plt.subplots();ax.plot(t_arr,lr_arr);ax.set(xlabel="normalized t",ylabel="latent representation")
fig.savefig(OUT / "latent.png");plt.show();plt.close(fig)
# The spatial basis and time coefficients reconstruct the same learned linear representation.
phi = model.model_x_to_phi()(tensor(train_data[:32,1:3]))
print("spatial basis shape",tuple(phi.shape))


# compute the reconstruction on training data

In [ ]:
input_pred = train_data[:,:3]
time_jump_index = np.r_[np.flatnonzero(np.diff(input_pred[:,0]) != 0)+1, len(input_pred)]
uv_pred = predict(model,input_pred)


## we can do a sanity check to see that the number of grid point is changing over time, so it is an AMR dataset 

In [ ]:
plt.figure(figsize=(6,6))
plt.plot(np.diff(time_jump_index),'k-o')
plt.xlabel('time',fontsize=20)
plt.ylabel('number of mesh point at that time',fontsize=20)
plt.savefig(OUT / "amr_mesh_counts.png");plt.show();plt.close()


# draw the reconstruction

In [ ]:
mean = tmp.mean
std = tmp.std

ti = 0
ti_ = time_jump_index[ti]

fig,axs=plt.subplots(2,2,figsize=(10.75,8))
circle1 = plt.Circle((0, 0), 0.0035, color='grey')
circle2 = plt.Circle((0, 0), 0.0035, color='grey')
circle3 = plt.Circle((0, 0), 0.0035, color='grey')
circle4 = plt.Circle((0, 0), 0.0035, color='grey')


umax = np.max(train_data[:ti_,3]*std[3]+mean[3])
umin = np.min(train_data[:ti_,3]*std[3]+mean[3])
vmax = np.max(train_data[:ti_,4]*std[4]+mean[4])
vmin = np.min(train_data[:ti_,4]*std[4]+mean[4])

axs[0,0].tricontourf(train_data[:ti_,1]*std[1]+mean[1], train_data[:ti_,2]*std[2]+mean[2], 
                     train_data[:ti_,3]*std[3]+mean[3],levels=50,vmin=umin,vmax=umax)
axs[0,1].tricontourf(train_data[:ti_,1]*std[1]+mean[1], train_data[:ti_,2]*std[2]+mean[2], 
                     train_data[:ti_,4]*std[4]+mean[4],levels=50,vmin=vmin,vmax=vmax)

axs[1,0].tricontourf(train_data[:ti_,1]*std[1]+mean[1], train_data[:ti_,2]*std[2]+mean[2], 
                     uv_pred[:ti_,0]*std[3]+mean[3],levels=50,vmin=umin,vmax=umax)
axs[1,1].tricontourf(train_data[:ti_,1]*std[1]+mean[1], train_data[:ti_,2]*std[2]+mean[2], 
                     uv_pred[:ti_,1]*std[4]+mean[4],levels=50,vmin=vmin,vmax=vmax)


for j in range(2):
    for i in range(2):
        axs[i,j].set_xlim([-0.01,0.04])
        axs[i,j].set_ylim([-0.02,0.02])
        axs[i,j].axis('equal')
    
axs[0,0].set_title('U - true',fontsize=20)
axs[0,1].set_title('V - true',fontsize=20)
axs[1,0].set_title('U - pred',fontsize=20)
axs[1,1].set_title('V - pred',fontsize=20)
axs[0,0].add_patch(circle1)
axs[0,1].add_patch(circle2)
axs[1,0].add_patch(circle3)
axs[1,1].add_patch(circle4)
fig.suptitle('demo of a very small NIF with 2 layers and 30 neurons', fontsize=30)
fig.savefig(OUT / "cylinder_reconstruction.png");plt.show();plt.close(fig)
per_row = np.mean((uv_pred-train_data[:,3:5])**2,axis=1)
write_metrics({"epochs":epochs(nepoch),"training_rows":training_rows,"all_rows":len(train_data),
               "weighted_mse":float(np.mean(per_row*np.asarray(sample_weight).reshape(-1))),
               "physical_u_mse":float(np.mean(((uv_pred[:,0]-train_data[:,3])*tmp.std[3])**2)),
               "physical_v_mse":float(np.mean(((uv_pred[:,1]-train_data[:,4])*tmp.std[4])**2))})


# Summary
- we learned how to learn a linear representation on spatial-temporal dynamics on arbitrary mesh 
- we demonstrated on a (shortened) flow over a cylinder case with a small neural network, just to give a flavour how it is trained and how does it look like